# 模式分割

In [ ]:
import sys
from pathlib import Path
ROOT = Path(".").resolve().parents[3]
sys.path.extend([f"{ROOT}/tests", f"{ROOT}/src"])
# # from tools.tag_span import _create_span, _set_span, _verify_structural_equal_with_span
from tools.torch_utils import verify_model

In [ ]:
import numpy as np

import tvm
from tvm import relay
from tvm.relay.build_module import bind_params_by_name
from tvm.relay.dataflow_pattern import *
from tvm.relay.testing import run_opt_pass

# NB: 1 corresponds to the C++ enum that specicfies this
# we loose the type safety due to the Python/C++ calling
# convention.
K_ELEMWISE = 0
K_BROADCAST = 1

In [ ]:

def test_double_partition():
    # Pattern 1
    conv2d_p = is_op("nn.conv2d")(wildcard(), wildcard())
    bias_add_p = is_op("nn.bias_add")(conv2d_p, wildcard())
    relu_p = is_op("nn.relu")(bias_add_p)

    # Graph
    x = relay.var("input")
    w = relay.var("weight")
    b = relay.var("bias")
    w2 = relay.var("weight")
    b2 = relay.var("bias")
    conv2d = relay.op.nn.conv2d(x, w)
    bias_add = relay.op.nn.bias_add(conv2d, b)
    relu = relay.op.nn.relu(bias_add)
    conv2d2 = relay.op.nn.conv2d(relu, w2)
    bias_add2 = relay.op.nn.bias_add(conv2d2, b2)

    partitioned = bias_add2
    for pat, label in [(relu_p, "conv_bias_relu"), (bias_add_p, "conv_bias")]:
        partitioned = pat.partition(partitioned, {"Composite": label})

    inpf = relay.var("input")
    weightf = relay.var("weight")
    biasf = relay.var("bias")
    func0 = (
        relay.Function(
            [inpf, weightf, biasf],
            relay.op.nn.relu(relay.op.nn.bias_add(relay.op.nn.conv2d(inpf, weightf), biasf)),
        )
        .with_attr("Composite", "conv_bias_relu")
        .with_attr("PartitionedFromPattern", "nn.conv2d_nn.bias_add_nn.relu_")
    )
    inpf = relay.var("input")
    weightf = relay.var("weight")
    biasf = relay.var("bias")
    func1 = (
        relay.Function(
            [inpf, weightf, biasf], relay.op.nn.bias_add(relay.op.nn.conv2d(inpf, weightf), biasf)
        )
        .with_attr("Composite", "conv_bias")
        .with_attr("PartitionedFromPattern", "nn.conv2d_nn.bias_add_")
    )

    expected = func1(func0(x, w, b), w2, b2)
    assert tvm.ir.structural_equal(partitioned, expected)


def test_partition_dominator():
    # Pattern
    is_conv2d = is_op("nn.conv2d")(wildcard(), wildcard())
    is_unary_elemwise = (wildcard().has_attr({"TOpPattern": K_ELEMWISE}))(wildcard())
    reduction = is_op("add")(wildcard(), wildcard())
    diamond = dominates(is_conv2d, is_unary_elemwise, reduction)

    # Classic Diamond
    inp = relay.var("input")
    weight = relay.var("weight")

    def generate_diamond(inp, weight):
        conv2d = relay.op.nn.conv2d(inp, weight)
        relu = relay.op.nn.relu(conv2d)
        relu = relay.op.nn.relu(relu)
        leaky_relu = relay.op.nn.leaky_relu(conv2d, alpha=0)
        return relu + leaky_relu

    out = generate_diamond(inp * inp, weight * weight)
    # Check
    partitioned = diamond.partition(out)

    i = relay.Var("input")
    w = relay.Var("weight")
    f = relay.Function([i, w], generate_diamond(i, w)).with_attr(
        "PartitionedFromPattern", "nn.conv2d_nn.relu_nn.relu_nn.leaky_relu_add_"
    )
    assert tvm.ir.structural_equal(partitioned, f(inp * inp, weight * weight))


def test_quadruple_partition_dominator():
    # Pattern
    is_conv2d = is_op("nn.conv2d")(wildcard(), wildcard())
    is_unary_elemwise = (wildcard().has_attr({"TOpPattern": K_ELEMWISE}))(wildcard()) | is_op(
        "add"
    )(wildcard(), wildcard())
    reduction = is_op("add")(wildcard(), wildcard())
    diamond = dominates(is_conv2d, is_unary_elemwise, reduction)

    inp = relay.var("input")
    weight = relay.var("weight")

    # Classic Diamond
    def classic_diamond(inp, weight):
        conv2d = relay.op.nn.conv2d(inp, weight)
        relu = relay.op.nn.relu(conv2d)
        relu = relay.op.nn.relu(relu)
        leaky_relu = relay.op.nn.leaky_relu(conv2d, alpha=0)
        return relu + leaky_relu

    # Deeper Branch
    def deeper_diamond(inp, weight):
        conv2d = relay.op.nn.conv2d(inp, weight)
        relu = relay.op.nn.relu(conv2d)
        relu = relay.op.nn.relu(relu)
        relu = relay.op.tanh(relu)
        leaky_relu = relay.op.nn.leaky_relu(conv2d, alpha=0)
        return relu + leaky_relu

    # Single Branch
    def single_branch(inp, weight):
        conv2d = relay.op.nn.conv2d(inp, weight)
        relu = relay.op.nn.relu(conv2d)
        relu = relay.op.nn.relu(relu)
        tanh = relay.op.tanh(relu)
        return relu + tanh

    # Fuzzy path/nested Diamond
    def nested_diamond(inp, weight):
        conv2d = relay.op.nn.conv2d(inp, weight)
        relu = relay.op.nn.relu(conv2d)
        relu = relu + relu
        tanh = relay.op.tanh(relu)
        leaky_relu = relay.op.nn.leaky_relu(conv2d, alpha=0)
        return tanh + leaky_relu

    partitioned = diamond.partition(
        nested_diamond(
            single_branch(deeper_diamond(classic_diamond(inp, weight), weight), weight), weight
        )
    )

    functions = []
    partition_names = [
        "nn.conv2d_nn.relu_nn.relu_nn.leaky_relu_add_",
        "nn.conv2d_nn.relu_nn.relu_tanh_nn.leaky_relu_add_",
        "nn.conv2d_nn.relu_nn.relu_tanh_add_",
        "nn.conv2d_nn.relu_add_tanh_nn.leaky_relu_add_",
    ]
    for i, f in enumerate([classic_diamond, deeper_diamond, single_branch, nested_diamond]):
        inpf = relay.var("input")
        weightf = relay.var("weight")
        functions.append(
            relay.Function([inpf, weightf], f(inpf, weightf)).with_attr(
                "PartitionedFromPattern", partition_names[i]
            )
        )

    reference = functions[3](
        functions[2](functions[1](functions[0](inp, weight), weight), weight), weight
    )
    assert tvm.ir.structural_equal(partitioned, reference)


def get_BN(x, var, mean, beta, gamma, eps):
    return gamma * (x - mean) / relay.op.sqrt(var + eps) + beta


def test_partition_batchnorm():
    x = relay.var("x")
    var = relay.var("var")
    mean = relay.var("mean")
    beta = relay.var("beta")
    gamma = relay.var("gamma")
    eps = relay.const(1e-5)
    BN = get_BN(x, var, mean, beta, gamma, eps)

    xf = relay.var("xf")
    varf = relay.var("varf")
    meanf = relay.var("meanf")
    betaf = relay.var("betaf")
    gammaf = relay.var("gammaf")
    # Put the arguments in toplogological order for the reference
    f = relay.Function(
        [gammaf, xf, meanf, varf, betaf], get_BN(xf, varf, meanf, betaf, gammaf, eps)
    ).with_attr("PartitionedFromPattern", "subtract_multiply_add_sqrt_divide_add_")

    partitioned = BatchnormCallback().pattern.partition(BN)
    reference = f(gamma, x, mean, var, beta)
    assert tvm.ir.structural_equal(partitioned, reference)


def test_partition_double_batchnorm():
    x = relay.var("x")
    var = relay.var("var")
    mean = relay.var("mean")
    beta = relay.var("beta")
    gamma = relay.var("gamma")
    eps = relay.const(1e-5)

    BN = gamma * (x - mean) / relay.op.sqrt(var + eps) + beta
    BN2 = gamma * (BN - mean) / relay.op.sqrt(var + eps) + beta

    xf = relay.var("xf")
    varf = relay.var("varf")
    meanf = relay.var("meanf")
    betaf = relay.var("betaf")
    gammaf = relay.var("gammaf")
    f1 = relay.Function(
        [gammaf, xf, meanf, varf, betaf], get_BN(xf, varf, meanf, betaf, gammaf, eps)
    ).with_attr("PartitionedFromPattern", "subtract_multiply_add_sqrt_divide_add_")
    # The partitioner doesn't replace duplicates, so we use two copies of the function
    xf2 = relay.var("xf2")
    varf2 = relay.var("varf2")
    meanf2 = relay.var("meanf2")
    betaf2 = relay.var("betaf2")
    gammaf2 = relay.var("gammaf2")
    f2 = relay.Function(
        [gammaf2, xf2, meanf2, varf2, betaf2], get_BN(xf2, varf2, meanf2, betaf2, gammaf2, eps)
    ).with_attr("PartitionedFromPattern", "subtract_multiply_add_sqrt_divide_add_")

    partitioned = BatchnormCallback().pattern.partition(BN2)
    reference = f2(gamma, f1(gamma, x, mean, var, beta), mean, var, beta)
    assert tvm.ir.structural_equal(partitioned, reference)


def test_overlappting_partitions():
    x = wildcard()
    gamma = wildcard()
    beta = wildcard()
    moving_mean = wildcard()
    moving_var = wildcard()
    bn_node = is_op("nn.batch_norm")(x, gamma, beta, moving_mean, moving_var)
    tuple_get_item_node = TupleGetItemPattern(bn_node, 0)

    x = relay.var("x")
    var = relay.var("var")
    mean = relay.var("mean")
    beta = relay.var("beta")
    gamma = relay.var("gamma")
    BN = relay.op.nn.batch_norm(x, gamma, beta, mean, var, epsilon=1e-5)
    T1 = BN[0]
    T2 = BN[0]
    add = T1 + T2

    assert tuple_get_item_node.partition(add) == add


def test_partition_overused():
    pattern = is_op("nn.relu")(is_op("nn.conv2d")(wildcard(), wildcard()))

    x = relay.var("input")
    w = relay.var("weight")
    conv2d = relay.op.nn.conv2d(x, w)
    relu = relay.op.nn.relu(conv2d)
    out = relu + conv2d

    assert pattern.partition(out) == out


def test_partition_fuzzy_tuple():
    x = relay.var("x")
    y = relay.var("y")
    z = x + y
    tuple_pattern = is_tuple(None)
    concat_pattern = is_op("concatenate")(tuple_pattern)

    xp = relay.var("xp")
    yp = relay.var("yp")
    zp = relay.var("zp")

    def create_func(args, body):
        return relay.Function(args, body).with_attr("PartitionedFromPattern", "Tuple_concatenate_")

    def concat(*args):
        return relay.op.concatenate(relay.expr.Tuple(args), axis=0)

    one = concat_pattern.partition(concat(x))
    assert tvm.ir.structural_equal(one, create_func([xp], concat(xp))(x))
    two = concat_pattern.partition(concat(x, y))
    assert tvm.ir.structural_equal(two, create_func([xp, yp], concat(xp, yp))(x, y))
    three = concat_pattern.partition(concat(x, y, z))
    assert tvm.ir.structural_equal(three, create_func([xp, yp, zp], concat(xp, yp, zp))(x, y, z))


def test_partition_fuzzy_function_args():
    func_pattern = FunctionPattern(None, wildcard() + wildcard())(None) + wildcard()
    x = relay.var("x")
    y = relay.var("y")
    z = relay.var("z")
    b = relay.var("b")
    xp = relay.var("xp")
    yp = relay.var("yp")
    zp = relay.var("zp")

    def create_func(call):
        N = len(call.op.params)
        new_params = [relay.var(str(i)) for i in range(N + 1)]
        label = "add_FunctionCall_add_"
        if N == 3:
            label = "add_" + label
        return relay.Function(
            new_params, relay.Call(call.op, (new_params[0:-1])) + new_params[-1]
        ).with_attr("PartitionedFromPattern", label)(*([x, y, z][0:N] + [b]))

    f1 = relay.Function([xp], xp + xp)(x)
    one = func_pattern.partition(f1 + b)
    assert tvm.ir.structural_equal(one, create_func(f1))
    f2 = relay.Function([xp, yp], xp + yp)(x, y)
    two = func_pattern.partition(f2 + b)
    assert tvm.ir.structural_equal(two, create_func(f2))
    f3 = relay.Function([xp, yp, zp], xp + yp + zp)(x, y, z)
    three = func_pattern.partition(f3 + b)
    assert tvm.ir.structural_equal(three, create_func(f3))


def test_partition_check():
    pattern = is_op("nn.relu")(is_op("nn.conv2d")(is_var("input"), wildcard()))

    def check(pre):
        return pre.args[0].attrs.data_layout == "NCHW"

    x = relay.var("input")
    w = relay.var("weight")
    conv2d = relay.op.nn.conv2d(x, w)
    relu = relay.op.nn.relu(conv2d)

    xf = relay.var("input")
    wf = relay.var("weight")
    conv2df = relay.op.nn.conv2d(xf, wf)
    reluf = relay.op.nn.relu(conv2df)
    func = relay.Function([xf, wf], reluf).with_attr("PartitionedFromPattern", "nn.conv2d_nn.relu_")

    reference = func(x, w)
    partitioned = pattern.partition(relu, check=check)
    assert tvm.ir.structural_equal(partitioned, reference)

    conv2d = relay.op.nn.conv2d(x, w, data_layout="NHWC")
    relu = relay.op.nn.relu(conv2d)
    assert relu == pattern.partition(relu, check=check)


def test_partition_check_types():
    pattern = is_op("nn.relu")(is_op("nn.conv2d")(wildcard(), wildcard()))

    def check(pre):
        conv = pre.args[0]
        return (conv.attrs.data_layout == "NCHW") and bool(conv.checked_type.shape[0] == 1)

    x = relay.var("input", shape=(1, 10, 10, 10))
    w = relay.var("weight", shape=(10, 10, 3, 3))
    conv2d = relay.op.nn.conv2d(x, w)
    relu = relay.op.nn.relu(conv2d)
    relu = run_opt_pass(relu, relay.transform.InferType())

    partitioned = pattern.partition(relu, check=check)
    assert partitioned.op.attrs["PartitionedFromPattern"] == "nn.conv2d_nn.relu_"

    conv2d = relay.op.nn.conv2d(x, w, data_layout="NHWC")
    relu = relay.op.nn.relu(conv2d)
    relu = run_opt_pass(relu, relay.transform.InferType())
    assert relu == pattern.partition(relu, check=check)

    x = relay.var("input", shape=(2, 10, 10, 10))
    w = relay.var("weight", shape=(10, 10, 3, 3))
    conv2d = relay.op.nn.conv2d(x, w)
    relu = relay.op.nn.relu(conv2d)
    relu = run_opt_pass(relu, relay.transform.InferType())
    assert relu == pattern.partition(relu, check=check)


def conv_bias_relu(x, w, b):
    conv2d = relay.op.nn.conv2d(x, w)
    bias_add = relay.op.nn.bias_add(conv2d, b)
    relu = relay.op.nn.relu(bias_add)
    return relu


def test_partition_option():
    x = relay.var("x")
    w = relay.var("w")
    b = relay.var("b")

    conv2d = is_op("nn.conv2d")(wildcard(), wildcard())
    bias = conv2d.optional(lambda x: is_op("nn.bias_add")(x, wildcard()))
    pattern1 = is_op("nn.relu")(bias)

    conv2d = is_op("nn.conv2d")(wildcard(), wildcard())
    bias = is_op("nn.bias_add")(conv2d, wildcard())
    pattern2 = bias.optional(lambda x: is_op("nn.relu")(x))

    relu = conv_bias_relu(x, w, b)

    xf = relay.var("x")
    wf = relay.var("w")
    bf = relay.var("b")
    func = relay.Function([xf, wf, bf], conv_bias_relu(xf, wf, bf)).with_attr(
        "PartitionedFromPattern", "nn.conv2d_nn.bias_add_nn.relu_"
    )

    assert pattern1.match(relu)
    assert tvm.ir.structural_equal(func(x, w, b), pattern1.partition(relu))

    assert pattern2.match(relu)
    assert tvm.ir.structural_equal(func(x, w, b), pattern2.partition(relu))


def test_partition_function():
    x = relay.var("x")
    w = relay.var("w")
    b = relay.var("b")

    x1 = relay.var("x1")
    w1 = relay.var("w1")

    wc_x = wildcard()
    wc_w = wildcard()
    wc_b = wildcard()
    wc_x1 = wildcard()
    wc_w1 = wildcard()

    func_pattern = FunctionPattern([wc_x1, wc_w1], is_op("nn.conv2d")(wc_x1, wc_w1))
    pattern = func_pattern(wc_x, wc_w) + wc_b

    func = relay.Function([x1, w1], relay.nn.conv2d(x1, w1))
    expr = func(x, w) + b + b

    x2 = relay.var("x2")
    w2 = relay.var("w2")
    b2 = relay.var("b2")
    func2 = relay.Function([x2, w2, b2], func(x2, w2) + b2).with_attr(
        "PartitionedFromPattern", "nn.conv2d_FunctionCall_add_"
    )
    expr2 = func2(x, w, b) + b
    assert tvm.ir.structural_equal(pattern.partition(expr), expr2)


def test_partition_optional_function():
    x = relay.var("x")
    w = relay.var("w")
    b = relay.var("b")

    x1 = relay.var("x1")
    w1 = relay.var("w1")

    wc_x = wildcard()
    wc_w = wildcard()
    wc_x1 = wildcard()
    wc_w1 = wildcard()

    func_pattern0 = FunctionPattern(
        [wc_x1, wc_w1], is_op("sigmoid")(is_op("nn.conv2d")(wc_x1, wc_w1))
    )
    func_pattern1 = FunctionPattern(
        [wc_x1, wc_w1], is_op("nn.relu")(is_op("nn.conv2d")(wc_x1, wc_w1))
    )
    pattern = func_pattern0(wc_x, wc_w) | func_pattern1(wc_x, wc_w)

    func = relay.Function([x1, w1], relay.nn.relu(relay.nn.conv2d(x1, w1)))
    expr = func(x, w) + b

    x2 = relay.var("x2")
    w2 = relay.var("w2")
    func2 = relay.Function([x2, w2], func(x2, w2)).with_attr(
        "PartitionedFromPattern", "nn.conv2d_nn.relu_FunctionCall_"
    )
    expr2 = func2(x, w) + b
    assert tvm.ir.structural_equal(pattern.partition(expr), expr2)


def test_rewrite_function_with_fuzzy_body():
    """Allow Rewriting a function with a fuzzy body via dominator analysis"""
    x = relay.var("x")
    w = relay.var("w")
    b = relay.var("b")

    x1 = relay.var("x1")
    w1 = relay.var("w1")

    wc_x = wildcard()
    wc_w = wildcard()
    wc_b = wildcard()
    wc_x1 = wildcard()
    wc_w1 = wildcard()

    func_pattern = FunctionPattern([wc_x1, wc_w1], wildcard())
    pattern = func_pattern(wc_x, wc_w) + wc_b

    func = relay.Function([x1, w1], relay.nn.conv2d(x1, w1))
    expr = func(x, w) + b + b

    class TestRewrite(DFPatternCallback):
        def __init__(self):
            super(TestRewrite, self).__init__()
            self.pattern = pattern

        def callback(self, pre, post, node_map):
            return x + w

    out = rewrite(TestRewrite(), expr)
    assert tvm.ir.structural_equal(out, x + w + b)


def test_partition_function_with_fuzzy_body():
    """
    Allow Rewriting a function with a fuzzy body via dominator analysis
    """
    x = relay.var("x")
    w = relay.var("w")
    b = relay.var("b")

    x1 = relay.var("x1")
    w1 = relay.var("w1")

    wc_x = wildcard()
    wc_w = wildcard()
    wc_b = wildcard()
    wc_x1 = wildcard()
    wc_w1 = wildcard()

    func_pattern = FunctionPattern([wc_x1, wc_w1], wildcard())
    pattern = func_pattern(wc_x, wc_w) + wc_b

    func = relay.Function([x1, w1], relay.nn.conv2d(x1, w1))
    expr = func(x, w) + b + b

    x2 = relay.var("x2")
    w2 = relay.var("w2")
    b2 = relay.var("b2")
    func2 = relay.Function([x2, w2, b2], func(x2, w2) + b2).with_attr(
        "PartitionedFromPattern", "nn.conv2d_FunctionCall_add_"
    )
    expr2 = func2(x, w, b) + b
    assert tvm.ir.structural_equal(pattern.partition(expr), expr2)


In [ ]:

def test_match_match():
    add_pattern = is_op("add")(wildcard(), wildcard())

    class TestRewrite(DFPatternCallback):
        def __init__(self):
            super(TestRewrite, self).__init__()
            self.pattern = add_pattern

        def callback(self, pre, post, node_map):
            return post.args[0] - post.args[1]

    mod = tvm.IRModule({})
    tvm.relay.prelude.Prelude(mod)
    # Apply rewrite on IR including relay.Match
    out = rewrite(TestRewrite(), mod["tensor_concatenate_int64"])
    assert tvm.ir.structural_equal(mod["tensor_concatenate_int64"], out)


def test_partition_constant_embedding():
    x = relay.var("x")
    w = relay.var("w")
    wc = relay.const(1)
    b = relay.var("b")

    xf = relay.var("x")
    wf = relay.var("w")
    bf = relay.var("b")
    embeded_func = relay.Function([xf, bf], conv_bias_relu(xf, wc, bf)).with_attr(
        "PartitionedFromPattern", "nn.conv2d_nn.bias_add_nn.relu_"
    )
    xf = relay.var("x")
    wf = relay.var("w")
    bf = relay.var("b")
    lifted_func = relay.Function([xf, wf, bf], conv_bias_relu(xf, wf, bf)).with_attr(
        "PartitionedFromPattern", "nn.conv2d_nn.bias_add_nn.relu_"
    )
    relu = conv_bias_relu(x, w, b)
    reluc = conv_bias_relu(x, wc, b)

    # Check lifting of wildcard matches
    pattern = is_op("nn.relu")(
        is_op("nn.bias_add")(is_op("nn.conv2d")(wildcard(), wildcard()), wildcard())
    )
    assert tvm.ir.structural_equal(lifted_func(x, w, b), pattern.partition(relu))
    assert tvm.ir.structural_equal(lifted_func(x, wc, b), pattern.partition(reluc))

    # Check lifting of input matches
    pattern = is_op("nn.relu")(
        is_op("nn.bias_add")(is_op("nn.conv2d")(wildcard(), is_var()), wildcard())
    )
    assert tvm.ir.structural_equal(lifted_func(x, w, b), pattern.partition(relu))
    assert tvm.ir.structural_equal(reluc, pattern.partition(reluc))  # Constants are not Inputs

    # Check embedding of constant matches
    pattern = is_op("nn.relu")(
        is_op("nn.bias_add")(is_op("nn.conv2d")(wildcard(), is_constant()), wildcard())
    )
    assert tvm.ir.structural_equal(relu, pattern.partition(relu))
    assert tvm.ir.structural_equal(embeded_func(x, b), pattern.partition(reluc))

    # Check embedding of constant ExprPatterns
    pattern = is_op("nn.relu")(
        is_op("nn.bias_add")(is_op("nn.conv2d")(wildcard(), is_expr(wc)), wildcard())
    )
    assert tvm.ir.structural_equal(relu, pattern.partition(relu))
    assert tvm.ir.structural_equal(embeded_func(x, b), pattern.partition(reluc))

    # Check lifting/embedding of Alt matches
    pattern = is_op("nn.relu")(
        is_op("nn.bias_add")(is_op("nn.conv2d")(wildcard(), is_var() | is_constant()), wildcard())
    )
    assert tvm.ir.structural_equal(lifted_func(x, w, b), pattern.partition(relu))
    assert tvm.ir.structural_equal(embeded_func(x, b), pattern.partition(reluc))

    # Check lifting/embedding of Alt matches with the other ordering
    pattern = is_op("nn.relu")(
        is_op("nn.bias_add")(is_op("nn.conv2d")(wildcard(), is_constant() | is_var()), wildcard())
    )
    assert tvm.ir.structural_equal(lifted_func(x, w, b), pattern.partition(relu))
    assert tvm.ir.structural_equal(embeded_func(x, b), pattern.partition(reluc))


def test_rewrite_once():
    # This class recursively removes the arguments to concat until there is nothing left to concatenate.
    class ConcatRewriter(DFPatternCallback):
        def __init__(self, rewrite_once):
            super().__init__(rewrite_once=rewrite_once)
            self.pattern = is_op("concatenate")(None)

        def callback(self, pre, post, node_map):
            concat_args = post.args[0]
            # Remove the last argument
            new_args = [concat_args[i] for i in range(len(concat_args) - 1)]
            if new_args:
                return relay.op.concatenate(relay.expr.Tuple(new_args), axis=0)
            else:
                return concat_args[0]

    x = relay.var("x")
    y = relay.var("y")
    z = relay.var("z")
    concat = relay.op.concatenate(relay.expr.Tuple([x, y, z]), axis=0)

    def test_one_callback():
        # Let the rewriter run recursively
        out = rewrite(ConcatRewriter(False), concat)
        expected = x
        assert tvm.ir.structural_equal(out, expected)

        # Run the rewriter once
        out = rewrite(ConcatRewriter(True), concat)
        expected = relay.op.concatenate(relay.expr.Tuple([x, y]), axis=0)
        assert tvm.ir.structural_equal(out, expected)

    def test_multi_callbacks():
        # This class recursively add a nn.relu operator after nn.softmax
        class OneMoreReluRewriter(DFPatternCallback):
            def __init__(self, rewrite_once):
                super().__init__(rewrite_once=rewrite_once)
                self.pattern = is_op("nn.softmax")(None)

            def callback(self, pre, post, node_map):
                return relay.nn.relu(post)

        def before():
            # Before:
            #    x    y    z
            #    |    |    |
            #       concat
            #         |
            #      softmax
            return relay.nn.softmax(concat)

        def once_concat():
            # ConcatRewrite once, OneMoreReluRewrite once
            # Expected:
            #   x    y
            #   |    |
            #   concat
            #      |
            #   softmax
            #      |
            #    relu
            return relay.nn.relu(
                relay.nn.softmax(relay.op.concatenate(relay.expr.Tuple([x, y]), axis=0))
            )

        def recursive_concat():
            # ConcatRewrite recursively, OneMoreReluRewrite once
            # Expected:
            #      x
            #      |
            #   softmax
            #      |
            #    relu
            return relay.nn.relu(relay.nn.softmax(x))

        # Run ConcatRewriter once, OneMoreReluRewriter once
        out = rewrite(
            [OneMoreReluRewriter(True), ConcatRewriter(True)],
            before(),
        )
        assert tvm.ir.structural_equal(out, once_concat())

        # Run ConcatRewriter recursively, OneMoreReluRewriter once
        out = rewrite(
            [OneMoreReluRewriter(True), ConcatRewriter(False)],
            before(),
        )
        assert tvm.ir.structural_equal(out, recursive_concat())

    test_one_callback()
    test_multi_callbacks()


def test_matched_outside_but_dominated():
    """In this example the pattern matches the nn.conv2d/add/multiply flow. Even though the
    add output is consumed by the sigmoid, the sigmoid itself is dominated by the multiply.
    So partitioning can proceed, all be it with a duplication of the add."""
    in_mod = tvm.relay.parse(
        """
        #[version = "0.0.5"]
        def @main(%data: Tensor[(16, 16, 32, 32), float16], %weight: Tensor[(32, 16, 3, 3), float16], %bias: Tensor[(32), float32]) -> Tensor[(16, 32, 32, 32), float32] {
          %0 = layout_transform(%data, src_layout="NCHW", dst_layout="NHWC");
          %1 = layout_transform(%weight, src_layout="OIHW", dst_layout="OHWI");
          %2 = expand_dims(%bias, axis=1, num_newaxis=2);
          %3 = expand_dims(%2, axis=0);
          %4 = nn.conv2d(%0, %1, padding=[1, 1, 1, 1], channels=32, kernel_size=[3, 3], data_layout="NHWC", kernel_layout="OHWI", out_dtype="float32");
          %5 = layout_transform(%3, src_layout="NCHW", dst_layout="NHWC");
          %6 = add(%4, %5);
          %7 = sigmoid(%6);
          %8 = multiply(%6, %7);
          layout_transform(%8, src_layout="NHWC", dst_layout="NCHW")
        }
        """
    )
    expected_mod = tvm.relay.parse(
        """
        #[version = "0.0.5"]
        def @main(%data: Tensor[(16, 16, 32, 32), float16], %weight: Tensor[(32, 16, 3, 3), float16], %bias: Tensor[(32), float32]) -> Tensor[(16, 32, 32, 32), float32] {
          %2 = expand_dims(%bias, axis=1, num_newaxis=2);
          %3 = expand_dims(%2, axis=0);
          %4 = layout_transform(%data, src_layout="NCHW", dst_layout="NHWC");
          %5 = layout_transform(%weight, src_layout="OIHW", dst_layout="OHWI");
          %6 = nn.conv2d(%4, %5, padding=[1, 1, 1, 1], channels=32, kernel_size=[3, 3], data_layout="NHWC", kernel_layout="OHWI", out_dtype="float32");
          %7 = layout_transform(%3, src_layout="NCHW", dst_layout="NHWC");
          %8 = add(%6, %7);
          %9 = sigmoid(%8);
          %10 = fn (%FunctionVar_0_0, %FunctionVar_0_1, %FunctionVar_0_2, %FunctionVar_0_3, PartitionedFromPattern="nn.conv2d_add_multiply_") {
            %0 = nn.conv2d(%FunctionVar_0_0, %FunctionVar_0_1, padding=[1, 1, 1, 1], channels=32, kernel_size=[3, 3], data_layout="NHWC", kernel_layout="OHWI", out_dtype="float32");
            %1 = add(%0, %FunctionVar_0_2);
            multiply(%1, %FunctionVar_0_3)
          };
          %11 = %10(%4, %5, %7, %9);
          layout_transform(%11, src_layout="NHWC", dst_layout="NCHW")
        }
        """
    )
    pattern = is_op("multiply")(
        is_op("add")(is_op("nn.conv2d")(wildcard(), wildcard()), wildcard()), wildcard()
    )
    actual_mod = tvm.IRModule.from_expr(pattern.partition(in_mod["main"]))
    actual_mod = relay.transform.InferType()(actual_mod)
    tvm.ir.assert_structural_equal(actual_mod, expected_mod)


def test_partition_parallel_branch_with_same_input():
    """In this example, conv2d's two consumer(add and multiply) on two different branches are
    merged into one partition, make sure that the partitioned function has no redundant parameters"""
    # Pattern
    path1 = is_op("multiply")(wildcard(), wildcard())
    path2 = is_op("add")(wildcard(), wildcard())
    pattern = is_op("add")(path1, path2)

    i = relay.Var("input")
    w = relay.Var("weight")
    l = relay.Var("left")
    r = relay.Var("right")

    conv2d = relay.op.nn.conv2d(i, w)
    branch1 = relay.multiply(l, conv2d)
    branch2 = relay.add(conv2d, r)
    add = relay.add(branch1, branch2)

    lf = relay.Var("leftf")
    mf = relay.Var("midf")
    rf = relay.Var("rightf")
    f = relay.Function([lf, mf, rf], (lf * mf) + (mf + rf)).with_attr(
        "PartitionedFromPattern", "multiply_add_add_"
    )

    partitioned = pattern.partition(add)
    reference = f(l, conv2d, r)
    assert tvm.ir.structural_equal(partitioned, reference)


def test_rewrite_with_pattern_recursion():
    data = relay.var("data", relay.TensorType((2, 8), "float32"))
    dense_weight = relay.const(np.zeros((4, 8)))
    feat = relay.nn.dense(data, dense_weight)
    feat = relay.cast(feat, "float32")
    feat = relay.cast(feat, "float32")
    feat = relay.cast(feat, "float32")
    feat = relay.cast(feat, "float32")
    feat = relay.cast(feat, "float32")
    oup = relay.cast(feat, "float32")

    expected = relay.nn.relu(oup)

    class TheRewrite(DFPatternCallback):
        def __init__(self, pattern):
            super(TheRewrite, self).__init__(rewrite_once=True)
            self.pattern = pattern

        def callback(self, pre, post, node_map):
            return relay.nn.relu(post)

    def test_reset_call_args():
        dense_pattern = is_op("nn.dense")(wildcard(), wildcard())
        wildcard_redirect = wildcard()
        the_pattern = is_op("cast")(wildcard_redirect)
        the_pattern2 = the_pattern | dense_pattern
        wildcard_redirect.redirect_to(the_pattern2)

        actual = rewrite(TheRewrite(the_pattern), oup)
        tvm.ir.assert_structural_equal(actual, expected)

    def test_reset_alt_left():
        dense_pattern = is_op("nn.dense")(wildcard(), wildcard())
        wildcard_redirect = wildcard()
        or_pattern = wildcard_redirect | dense_pattern
        the_pattern = is_op("cast")(or_pattern)
        wildcard_redirect.redirect_to(the_pattern)

        actual = rewrite(TheRewrite(the_pattern), oup)
        tvm.ir.assert_structural_equal(actual, expected)

    def test_reset_alt_right():
        dense_pattern = is_op("nn.dense")(wildcard(), wildcard())
        wildcard_redirect = wildcard()
        or_pattern = dense_pattern | wildcard_redirect
        the_pattern = is_op("cast")(or_pattern)
        wildcard_redirect.redirect_to(the_pattern)

        actual = rewrite(TheRewrite(the_pattern), oup)
        tvm.ir.assert_structural_equal(actual, expected)

    test_reset_call_args()
    test_reset_alt_left()
    test_reset_alt_right()
